In [9]:
import pandas as pd
import numpy as np
from codelib.file_management.dynamic_file_pathing import get_root
import os

In [10]:
root = get_root()

data_folder = os.path.join(root, 'Simulation', 'Bonds', 'Data')

In [11]:
def prepare_data(
    destr_file,
    spot_curve_file,
    short_rate_column='Rate',
    spot_columns=['1Y', '5Y', '10Y', '20Y', '30Y'],
    spot_maturities=[1, 5, 10, 20, 30],
    frequency='M'  # 'M' = end-of-month; 'W' = weekly; 'BMS' = beginning of month
):
    """
    Load and align short rate (e.g. DESTR) and spot curve data from Excel files.

    Parameters:
        destr_file : str
            Path to the short-rate Excel file.
        spot_curve_file : str
            Path to the spot rate Excel file.
        short_rate_column : str
            Column name for the short rate.
        spot_columns : list
            Column names for spot curve maturities (e.g., ['1Y', '5Y', ...]).
        spot_maturities : list
            Maturity in years for each column in spot_columns.
        frequency : str
            Pandas resample frequency (e.g., 'M' for monthly).

    Returns:
        pd.DataFrame
            DataFrame with aligned short rate and zero-coupon prices.
    """
    # Load DESTR Excel
    destr = pd.read_excel(destr_file, parse_dates=True, index_col=0)
    destr = destr[[short_rate_column]].resample(frequency).last()

    # Load spot curve Excel
    spot = pd.read_excel(spot_curve_file, parse_dates=True, index_col=0)
    spot = spot[spot_columns].resample(frequency).last()

    # Align the data
    merged = destr.join(spot, how='inner')

    # Convert spot rates to zero-coupon bond prices
    for col, tau in zip(spot_columns, spot_maturities):
        price_col = f'ZCB_{tau}Y'
        merged[price_col] = np.exp(-merged[col] * tau)

    # Return final DataFrame
    zcb_cols = [f'ZCB_{tau}Y' for tau in spot_maturities]
    final = merged[[short_rate_column] + zcb_cols].dropna()
    return final

In [12]:
if __name__ == '__main__':
    destr_file = os.path.join(data_folder, "short_rate.xlsx")           # Your DESTR daily CSV with columns: ['Date', 'DESTR']
    spot_curve_file = os.path.join(data_folder, "1_to_30_Y.xlsx") # ECB spot rates CSV: ['Date', '1Y', '5Y', '10Y', ...]

    df = prepare_data(destr_file, spot_curve_file)
    print(df.head())

C:\Users\thorb\AppData\Local\Temp\ipykernel_28304\1606736935.py:32: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  destr = destr[[short_rate_column]].resample(frequency).last()


              Rate    ZCB_1Y    ZCB_5Y   ZCB_10Y   ZCB_20Y   ZCB_30Y
Date                                                                
2004-09-30  0.0209  0.977442  0.846638  0.665931  0.393410  0.230712
2004-10-31  0.0209  0.978066  0.854351  0.675263  0.399957  0.234664
2004-11-30  0.0210  0.978438  0.858300  0.682301  0.414057  0.250206
2004-12-31  0.0221  0.977816  0.858054  0.690362  0.424828  0.258819
2005-01-31  0.0209  0.978352  0.862750  0.700759  0.446302  0.283049


C:\Users\thorb\AppData\Local\Temp\ipykernel_28304\1606736935.py:36: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  spot = spot[spot_columns].resample(frequency).last()


In [13]:
df.to_excel(os.path.join(data_folder, 'prepared_data.xlsx'))